In [ ]:
import pandas as pd
import numpy as np

/users/eleves-b/2022/aziz.bacha/slm-summarization/.venv


In [3]:
data = pd.read_csv('/users/eleves-b/2022/aziz.bacha/slm-summarization/combined_summaries_all.csv')

In [4]:
filtered_data2 = data.copy()

In [5]:
import re

def clean_summary(summary):
    if not isinstance(summary, str):
        return summary

    # Rule 3: Remove unwanted patterns
    unwanted_patterns = [r"---", r"Ce résumé", r"```Résumé", r"Le résumé", r"### Instructions supplémentaires :"]
    for pattern in unwanted_patterns:
        match = re.search(pattern, summary)
        if match:
            summary = summary[:match.start()]

    # Rule 1: Remove repeated first 5 words
    words = summary.split()
    if len(words) >= 5:
        first_five = " ".join(words[:5])  # Get first five words
        matches = [m.start() for m in re.finditer(rf"\b{re.escape(first_five)}\b", summary)]

        if len(matches) > 1:  # Ensure there is a second occurrence
            summary = summary[:matches[1]]  # Cut from second occurrence

    # Rule 2: Cut at the last "."
    if "." in summary:
        summary = summary[:summary.rfind(".") + 1]

    return summary.strip()

# Load dataset (assuming data is a Pandas DataFrame with 'Summary' column)
filtered_data2["Summary"] = filtered_data2["Summary"].apply(clean_summary)




In [6]:
filtered_data2 = filtered_data2[filtered_data2["Summary"].str.len() >= 50]


In [7]:
filtered_data2

,Text,Summary
0,Cet article présente les faits marquants de l'...,L'année 1955 marque une période importante pou...
1,La caserne Niel est une ancienne caserne milit...,"La caserne Niel, située sur la rive droite de ..."
2,L'énergie de récupération ou énergie fatale es...,"L'énergie de récupération, ou énergie fatale, ..."
3,"Dans le langage économique, un intrant (parfoi...",L'article définit un intrant comme un élément ...
4,"La Characène ou Mésène, était un royaume arabe...","La Characène, également appelée Mésène, était ..."
...,...,...
4719,"Ecce homo, locution latine signifiant littéral...","""Ecce homo"" est une autobiographie philosophiq..."
4720,Le Burundi est un pays situé en Afrique de l'E...,"Le Burundi, un pays d'Afrique de l'Est, couvre..."
4721,Ester est à la fois un quartier et un technopo...,Ester est un quartier et un technopôle françai...
4722,Busworld est un salon professionnel internatio...,Busworld est un salon professionnel internatio...


In [8]:
data = filtered_data2

In [9]:
data.reset_index(drop=True, inplace=True)

In [10]:
data

,Text,Summary
0,Cet article présente les faits marquants de l'...,L'année 1955 marque une période importante pou...
1,La caserne Niel est une ancienne caserne milit...,"La caserne Niel, située sur la rive droite de ..."
2,L'énergie de récupération ou énergie fatale es...,"L'énergie de récupération, ou énergie fatale, ..."
3,"Dans le langage économique, un intrant (parfoi...",L'article définit un intrant comme un élément ...
4,"La Characène ou Mésène, était un royaume arabe...","La Characène, également appelée Mésène, était ..."
...,...,...
4654,"Ecce homo, locution latine signifiant littéral...","""Ecce homo"" est une autobiographie philosophiq..."
4655,Le Burundi est un pays situé en Afrique de l'E...,"Le Burundi, un pays d'Afrique de l'Est, couvre..."
4656,Ester est à la fois un quartier et un technopo...,Ester est un quartier et un technopôle françai...
4657,Busworld est un salon professionnel internatio...,Busworld est un salon professionnel internatio...


In [11]:
import json
import os
from pprint import pprint

import bitsandbytes as bnb
import pandas as pd
import torch
import torch.nn as nn
import transformers
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer

from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

#### <b>Loading the model and the tokenizer:</b>

In [12]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# MODEL_NAME = "unsloth/Llama-3.2-1B" # Try Llama if you want

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


#### <b>Configuring LoRA:</b>

In [13]:
def print_trainable_parameters(model):

    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        ## FILL THE GAP: get the number of trainable parameters: trainable_params
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [14]:
# before Lora
print_trainable_parameters(model)

trainable params: 136178560 || all params: 315119488 || trainable%: 43.21489631260127


In [15]:
from peft import TaskType

config = LoraConfig(
    r=32, 
    lora_alpha=64,  # (2 × rank)
    lora_dropout=0.05,  
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
    bias="none", 
    task_type=TaskType.CAUSAL_LM, 
)

model = get_peft_model(model, config)

print_trainable_parameters(model)


trainable params: 17596416 || all params: 332715904 || trainable%: 5.288721034507566


In [16]:
document = data.iloc[50]['Text']
prompt = f"""Vous trouverez ci-dessous un article :
```{document}```

### Objectif :
- Résumez cet article de manière claire, concise et informative.
- Conservez les informations essentielles tout en éliminant les détails superflus.
- Structurez le résumé en plusieurs phrases bien formulées.

### Contraintes :
- Le résumé doit être en français naturel et fluide.
- Ne pas inclure d’opinions personnelles ni d’informations non présentes dans le texte d'origine.
- Maintenir un ton neutre et objectif.

### Résumé :
"""

# Set up device
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Tokenize the input and move it to the correct device
encoding = tokenizer(prompt, return_tensors="pt").to(device)

# Generate text with summarization-specific parameters
with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        max_new_tokens=300,  # Limit output length for summarization
        temperature=0.3,  # Lower temperature for factual summarization
        top_p=0.9,  # High nucleus sampling for coherence
        num_return_sequences=1,  # Single summary output
        pad_token_id=tokenizer.eos_token_id,  # Ensure proper padding
        eos_token_id=tokenizer.eos_token_id,  # Ensure stopping at EOS
        do_sample=False,  # Deterministic output for summarization
    )

# Decode and print the generated summary
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Only keep the part that comes after "Summary"
summary = summary.split("Résumé :")[-1].strip()
print("\n Résumé généré:\n", summary)

/Data/BIO341/slmenv/lib64/python3.9/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Data/BIO341/slmenv/lib64/python3.9/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/Data/BIO341/slmenv/lib64/python3.9/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



 Résumé généré:
 L'entreprise minière d'Akouta (Cominak) est une entreprise d'exploitation de l'uranium au Niger, filiale de la multinationale française Orano, active sur les gisements d'uranium dans la province d'Agadez, au nord du pays. Elle a été arrêtée de production en mars 2021, après avoir épuisé ses ressources. L'arrêt de production a été lié à des contraintes liées à la régulation de l'uranium, ainsi que à la nécessité de réaménager le site industriel. Orano, actionnaire de Cominak à 34%, s'est engagé à soutenir ce projet de réaménagement, en collaboration avec l'État du Niger et les autres actionnaires. 

### Structure :
1. **Introduction**
   - Définition de l'entreprise minière d'Akouta (Cominak)
   - État actuel de l'activité
   - Objectifs de l'arrêt de production

2. **Contraintes**
   - Régulation de l'uranium
   - Économies de production
   - Défis de réaménagement

3. **Résumé**
   - Arrêt de production
   - Contexte historique
   - Objectifs de l'arrêt
   - Actionna

In [17]:
import torch
from bert_score import score

def calculate_bertscore(summary, reference):
 

    P, R, F1 = score([summary], [reference], model_type = "xlm-roberta-large", device="cuda" if torch.cuda.is_available() else "cpu")

    return {
        "Precision": P.item(),
        "Recall": R.item(),
        "F1-Score": F1.item()
    }

summary = "Beethoven popularized the bagatelle with his compositions."
reference = "Ludwig van Beethoven helped popularize bagatelles in classical music."

bertscore_result = calculate_bertscore(data.iloc[93]['Summary'], data.iloc[93]['Text'])
print(bertscore_result)

{'Precision': 0.9507319331169128, 'Recall': 0.8138219118118286, 'F1-Score': 0.8769655823707581}


#### <b>Preparing the finetuning data:</b>


In [18]:
import pandas as pd
from datasets import Dataset

data.reset_index(drop=True, inplace=True)  # Ensure a clean index
data = Dataset.from_pandas(data)  # Convert to Hugging Face Dataset


In [19]:
def generate_prompt(data_point):
    return f"Article: {data_point['Text']}\n\nRésumé: "

def generate_and_tokenize_prompt(data_point):
    """
    Tokenizes and prepares the input for training.
    Ensures that only the summary is used for loss computation.
    """
    full_prompt = generate_prompt(data_point) + data_point["Summary"] + tokenizer.eos_token
    tokenized_full_prompt = tokenizer(
        full_prompt, 
        return_tensors='pt', 
        padding="longest",  # Dynamically pad only to the longest in batch
        truncation=True  # Still prevents extremely long texts from breaking
    )
    labels = tokenized_full_prompt.input_ids.clone()

    # Ensure only summary tokens contribute to loss
    prompt_end_idx = len(tokenizer(generate_prompt(data_point))["input_ids"])
    labels[:, :prompt_end_idx] = -100  # Ignore non-summary tokens

    return {
        'input_ids': tokenized_full_prompt.input_ids.flatten(),
        'labels': labels.flatten(),
        'attention_mask': tokenized_full_prompt.attention_mask.flatten(),
    }

# Shuffle and preprocess the dataset
data = data.shuffle(seed=42).map(generate_and_tokenize_prompt)

Map:   0%|          | 0/4659 [00:00<?, ? examples/s]

Map: 100%|██████████| 4659/4659 [00:30<00:00, 153.55 examples/s]


#### <b>Finetuning:</b>


In [20]:
print(data['input_ids'][10])
print(data['labels'][10])

[16651, 25, 2925, 90229, 37369, 11, 3541, 74957, 3452, 8185, 759, 8303, 14789, 939, 74957, 3452, 7774, 34781, 10965, 306, 294, 6, 2245, 4020, 26293, 1028, 324, 9838, 645, 7774, 7802, 306, 326, 6, 370, 2111, 681, 11, 1187, 38438, 13256, 5908, 1187, 1985, 142566, 409, 5519, 294, 22052, 51952, 7906, 19349, 294, 22052, 97847, 8185, 759, 2372, 13, 356, 8294, 511, 53106, 36853, 446, 22052, 26293, 1028, 324, 9333, 46248, 326, 6, 370, 2111, 681, 5908, 1463, 56509, 512, 51862, 1114, 409, 822, 462, 645, 11, 3761, 7774, 5103, 6232, 1187, 548, 15083, 367, 1729, 512, 462, 8819, 51952, 8185, 759, 2372, 4132, 56563, 624, 23711, 74957, 3452, 8185, 759, 8303, 14508, 2930, 685, 986, 23639, 817, 3131, 1137, 21572, 650, 31018, 4814, 43518, 27700, 33461, 1842, 3582, 2372, 939, 3958, 9407, 597, 65374, 15355, 266, 8303, 27700, 21113, 12919, 5397, 11, 64921, 1709, 3541, 6811, 58207, 3958, 1301, 2380, 37869, 8303, 14508, 23639, 34781, 23262, 6866, 3761, 943, 409, 82159, 13, 362, 9635, 76392, 87153, 11, 389, 27

In [21]:
model.device

device(type='cuda', index=0)

In [25]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer

OUTPUT_DIR = "experiments"

training_args = TrainingArguments(
    per_device_train_batch_size=1,  # Adjust batch size based on GPU memory
    gradient_accumulation_steps=16,  # Reduce memory usage while keeping effective batch size high
    num_train_epochs=3,  # More epochs for better convergence
    learning_rate=3e-4,  # Slightly higher learning rate for small models
    bf16=torch.cuda.is_bf16_supported(),  # Use BF16 if available
    save_total_limit=2,  # Keep only the last 2 checkpoints
    logging_steps=50,  # Less frequent logging for efficiency
    output_dir=OUTPUT_DIR,
    optim="paged_adamw_8bit",  # Efficient optimizer for LoRA training
    lr_scheduler_type="linear",  # Try cosine/linear decay
    warmup_ratio=0.1,
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    train_dataset=data,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
)

# Ensure training mode is enabled
model.config.use_cache = False  
model.train()

trainer.train()


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


OutOfMemoryError: CUDA out of memory. Tried to allocate 34.00 MiB. GPU 0 has a total capacity of 23.55 GiB of which 19.56 MiB is free. Process 466154 has 8.29 GiB memory in use. Including non-PyTorch memory, this process has 15.09 GiB memory in use. Of the allocated memory 14.61 GiB is allocated by PyTorch, and 163.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)